# Building Final DataFrames for EDA and A/B Testing

## Objective

The purpose of this notebook is to build the final merged DataFrames that will be used for the Vanguard A/B testing project.

The cleaned input files are already available, so this notebook focuses on combining them, creating behavioral features, and saving analysis-ready datasets for EDA, KPI analysis, hypothesis testing, Tableau, and the final presentation.

The project combines three types of data:

- Client profile data
- Digital behavior data
- Experiment group assignment data

Because these datasets do not all have the same level of detail, we will create several final DataFrames instead of relying on only one.

---

## Folder Structure

This notebook is stored in:

`2_Python_Notebook/`

The cleaned input files are stored in:

`1.1_Clean_files/`

The final merged files created in this notebook will be saved in:

`1.2_Merged_Files_For_EDA/`

The relative paths used in this notebook will therefore be:

```python
input_path = "../1.1_Clean_files/"
output_path = "../1.2_Merged_Files_For_EDA/"
```

---

## Input Files

The notebook uses the following cleaned files:

| File | Description |
|---|---|
| `df1_demo_clean.csv` | Client profile data: age, gender, tenure, accounts, balance, calls, and logons |
| `df4_final_web_data_clean.csv` | Web activity data: visits, process steps, timestamps, and user interactions |
| `df_final_experiment_clients_clean.csv` | Experiment roster: identifies whether each client belongs to Test, Control, or empty |

---

## Why We Need Different DataFrames

The source files have different levels of granularity.

The client profile file is at **client level**, meaning each row represents one client.

The web activity file is at **event level**, meaning each row represents one interaction in the digital process. Therefore, the same client can appear multiple times.

The experiment file is also at **client level**, because each client belongs to one experiment group.

For this reason, using one single merged DataFrame for every analysis could create misleading results. For example, if client data is merged directly with event-level web data, highly active clients would appear more often and could distort averages, percentages, and hypothesis tests.

To avoid this, this notebook creates final DataFrames at different levels of analysis.

---

## Final DataFrames Created

### `event_big_df`

Event-level DataFrame.

Each row represents one web interaction.

Used for:

- Digital journey analysis
- Step-level analysis
- Tableau process-flow visualizations
- Exploring repeated steps, backtracking, and user movement through the process

This DataFrame should not be used as the main base for A/B hypothesis testing because clients can appear multiple times.

---

### `session_big_df`

Session-level DataFrame.

Each row represents one visit/session, grouped by:

- `client_id`
- `visitor_id`
- `visit_id`

Used for:

- Completion rate analysis
- Session duration analysis
- Backtracking analysis
- Repeated step analysis
- Friction analysis
- Main KPI analysis

This is one of the main DataFrames for evaluating process performance.

---

### `session_ab_df`

Session-level DataFrame filtered to include only:

- `Test`
- `Control`

Clients with `empty` variation are excluded.

Used for:

- Test vs Control comparison at session level
- Completion rate by variation
- Friction rate by variation
- Backtracking and repeated step comparison
- Session-level A/B performance metrics

---

### `client_big_df`

Client-level DataFrame.

Each row represents one client and includes demographic data plus aggregated behavior across sessions.

Used for:

- Demographic analysis
- Client segmentation
- Behavior by age, gender, tenure, balance, and number of accounts
- Client-level completion and friction analysis

---

### `client_ab_df`

Client-level DataFrame filtered to include only:

- `Test`
- `Control`

Used for:

- Client-level A/B testing
- Checking if Test and Control groups are balanced
- Comparing experiment results without duplicating clients

This is useful for the most statistically reliable experiment comparison because each client appears only once.

---

### `clean_sessions_df`

Contains only sessions that followed the ideal process flow:

`start → step_1 → step_2 → step_3 → confirm`

Used for:

- Analyzing successful user journeys
- Understanding smooth completions
- Comparing clean sessions between Test and Control

---

### `friction_sessions_df`

Contains sessions that did not follow the ideal process flow.

A session is considered to have friction if it includes abandonment, repeated steps, backtracking, skipped steps, or any non-linear journey.

Used for:

- Studying problematic journeys
- Comparing friction between Test and Control
- Identifying potential usability issues in the new interface

The term `friction` is preferred over `error`, because not every irregular journey is necessarily a technical error.

---

## Methodological Note

The A/B testing should not be based directly on the event-level DataFrame, because that would overrepresent clients with more interactions.

Instead:

- `session_ab_df` will be used for session-level performance metrics.
- `client_ab_df` will be used for client-level comparisons and hypothesis testing.
- `event_big_df` will mainly be used for process-flow exploration and Tableau visualizations.

---

## Expected Output Files

The following files will be saved into `1.2_Merged_Files_For_EDA/`:

| Output file | Purpose |
|---|---|
| `event_big_df.csv` | Event-level merged dataset |
| `session_big_df.csv` | Session-level merged dataset |
| `session_ab_df.csv` | Session-level Test vs Control dataset |
| `client_big_df.csv` | Client-level merged dataset |
| `client_ab_df.csv` | Client-level Test vs Control dataset |
| `clean_sessions_df.csv` | Sessions with a clean process flow |
| `friction_sessions_df.csv` | Sessions with friction or non-linear behavior |

In [1]:
import pandas as pd
import numpy as np
from pathlib import Path

input_path = Path("../1.1_Clean_files/")
output_path = Path("../1.2_Merged_Files_For_EDA/")

output_path.mkdir(parents=True, exist_ok=True)

In [2]:
df_demo = pd.read_csv(input_path / "df1_demo_clean.csv")
df_web = pd.read_csv(input_path / "df4_final_web_data_clean.csv")
df_exp = pd.read_csv(input_path / "df_final_experiment_clients_clean.csv")

In [3]:
df_demo.columns = df_demo.columns.str.strip().str.lower()
df_web.columns = df_web.columns.str.strip().str.lower()
df_exp.columns = df_exp.columns.str.strip().str.lower()

df_web["date_time"] = pd.to_datetime(df_web["date_time"], errors="coerce")

df_demo["gender"] = df_demo["gender"].astype(str).str.strip()
df_exp["variation"] = df_exp["variation"].astype(str).str.strip()

In [4]:
print("df_demo shape:", df_demo.shape)
print("df_web shape:", df_web.shape)
print("df_exp shape:", df_exp.shape)

print("\ndf_demo columns:")
print(df_demo.columns.tolist())

print("\ndf_web columns:")
print(df_web.columns.tolist())

print("\ndf_exp columns:")
print(df_exp.columns.tolist())

df_demo shape: (70595, 9)
df_web shape: (744641, 5)
df_exp shape: (70609, 2)

df_demo columns:
['client_id', 'client_tenure_years', 'client_tenure_months', 'client_age', 'gender', 'number_accounts', 'balance', 'calls_6_months', 'logons_6_months']

df_web columns:
['client_id', 'visitor_id', 'visit_id', 'process_step', 'date_time']

df_exp columns:
['client_id', 'variation']


In [5]:
print("Duplicate client_id in df_demo:", df_demo["client_id"].duplicated().sum())
print("Duplicate client_id in df_exp:", df_exp["client_id"].duplicated().sum())

print("\nMissing values in df_demo:")
print(df_demo.isna().sum())

print("\nMissing values in df_web:")
print(df_web.isna().sum())

print("\nMissing values in df_exp:")
print(df_exp.isna().sum())

print("\nExperiment variation counts:")
print(df_exp["variation"].value_counts(dropna=False))

Duplicate client_id in df_demo: 0
Duplicate client_id in df_exp: 0

Missing values in df_demo:
client_id               0
client_tenure_years     0
client_tenure_months    0
client_age              1
gender                  0
number_accounts         0
balance                 0
calls_6_months          0
logons_6_months         0
dtype: int64

Missing values in df_web:
client_id       0
visitor_id      0
visit_id        0
process_step    0
date_time       0
dtype: int64

Missing values in df_exp:
client_id    0
variation    0
dtype: int64

Experiment variation counts:
variation
Test       26968
Control    23532
empty      20109
Name: count, dtype: int64


In [6]:
demo_clients = set(df_demo["client_id"])
web_clients = set(df_web["client_id"])
exp_clients = set(df_exp["client_id"])

print("Unique clients in df_demo:", df_demo["client_id"].nunique())
print("Unique clients in df_web:", df_web["client_id"].nunique())
print("Unique clients in df_exp:", df_exp["client_id"].nunique())

print("\nClients in demo and web:", len(demo_clients & web_clients))
print("Clients in experiment and web:", len(exp_clients & web_clients))
print("Clients in experiment and demo:", len(exp_clients & demo_clients))

print("\nClients in experiment but not in demo:", len(exp_clients - demo_clients))
print("Clients in experiment but not in web:", len(exp_clients - web_clients))
print("Clients in web but not in experiment:", len(web_clients - exp_clients))

Unique clients in df_demo: 70595
Unique clients in df_web: 120157
Unique clients in df_exp: 70609

Clients in demo and web: 70595
Clients in experiment and web: 70609
Clients in experiment and demo: 70595

Clients in experiment but not in demo: 14
Clients in experiment but not in web: 0
Clients in web but not in experiment: 49548


In [7]:
print("Process step values:")
print(df_web["process_step"].value_counts(dropna=False).sort_index())

print("\nDate range:")
print("Min date:", df_web["date_time"].min())
print("Max date:", df_web["date_time"].max())

Process step values:
process_step
confirm    102506
start      234999
step_1     162797
step_2     132750
step_3     111589
Name: count, dtype: int64

Date range:
Min date: 2017-03-15 00:03:03
Max date: 2017-06-20 23:59:57


# Starting the merge: 


In [8]:
# =========================
# Create event-level big dataframe
# =========================

event_big_df = (
    df_web
    .merge(df_exp, on="client_id", how="left", validate="many_to_one")
    .merge(df_demo, on="client_id", how="left", validate="many_to_one")
)

event_big_df.shape

(744641, 14)

In [9]:
print("event_big_df shape:", event_big_df.shape)
print("Unique clients:", event_big_df["client_id"].nunique())
print("Unique visits:", event_big_df["visit_id"].nunique())

print("\nVariation counts at event level:")
print(event_big_df["variation"].value_counts(dropna=False))

print("\nMissing values:")
print(event_big_df.isna().sum())

event_big_df shape: (744641, 14)
Unique clients: 120157
Unique visits: 158095

Variation counts at event level:
variation
NaN        300744
Test       176699
Control    140536
empty      126662
Name: count, dtype: int64

Missing values:
client_id                    0
visitor_id                   0
visit_id                     0
process_step                 0
date_time                    0
variation               300744
client_tenure_years     300857
client_tenure_months    300857
client_age              300869
gender                  300857
number_accounts         300857
balance                 300857
calls_6_months          300857
logons_6_months         300857
dtype: int64


In [10]:
missing_variation_events = event_big_df["variation"].isna().sum()
missing_variation_clients = event_big_df.loc[event_big_df["variation"].isna(), "client_id"].nunique()

print("Events without variation:", missing_variation_events)
print("Clients without variation:", missing_variation_clients)

Events without variation: 300744
Clients without variation: 49548


In [11]:
missing_demo_events = event_big_df["client_age"].isna().sum()
missing_demo_clients = event_big_df.loc[event_big_df["client_age"].isna(), "client_id"].nunique()

print("Events without demographic data:", missing_demo_events)
print("Clients without demographic data:", missing_demo_clients)

Events without demographic data: 300869
Clients without demographic data: 49563


In [12]:
# All experiment clients, including empty
event_experiment_df = event_big_df[event_big_df["variation"].isin(["Test", "Control", "empty"])].copy()

# Only true A/B test clients
event_ab_df = event_big_df[event_big_df["variation"].isin(["Test", "Control"])].copy()

print("event_big_df:", event_big_df.shape)
print("event_experiment_df:", event_experiment_df.shape)
print("event_ab_df:", event_ab_df.shape)

event_big_df: (744641, 14)
event_experiment_df: (443897, 14)
event_ab_df: (317235, 14)


In [13]:
event_summary = (
    event_ab_df
    .groupby("variation")
    .agg(
        events=("process_step", "count"),
        clients=("client_id", "nunique"),
        visits=("visit_id", "nunique"),
        avg_events_per_client=("process_step", lambda x: x.count() / event_ab_df.loc[x.index, "client_id"].nunique()),
        avg_events_per_visit=("process_step", lambda x: x.count() / event_ab_df.loc[x.index, "visit_id"].nunique())
    )
    .round(2)
)

event_summary

,events,clients,visits,avg_events_per_client,avg_events_per_visit
variation,,,,,
Control,140536,23532,32189,5.97,4.37
Test,176699,26968,37136,6.55,4.76


In [14]:
event_big_df.to_csv(output_path / "event_big_df.csv", index=False)
event_experiment_df.to_csv(output_path / "event_experiment_df.csv", index=False)
event_ab_df.to_csv(output_path / "event_ab_df.csv", index=False)

## Event-Level DataFrame

The first merged dataset created in this notebook is `event_big_df`.

This DataFrame keeps the original event-level structure of the web activity data, where each row represents one interaction in the digital process. The experiment group and client profile data were added using `client_id`.

This dataset is useful for analyzing the full digital journey, process steps, repeated interactions, and Tableau visualizations. However, it will not be used as the main base for hypothesis testing because clients with more activity appear multiple times.

In [15]:
# =========================
# Define process step order
# =========================

step_order = {"start": 0, "step_1": 1, "step_2": 2, "step_3": 3, "confirm": 4}

df_web["step_num"] = df_web["process_step"].map(step_order)

# Check if any process_step was not mapped correctly
print("Unmapped process steps:", df_web["step_num"].isna().sum())

df_web_sorted = df_web.sort_values(["client_id", "visitor_id", "visit_id", "date_time", "step_num"]).copy()

Unmapped process steps: 0


In [16]:
# =========================
# Detect step movement within each session
# =========================

group_cols = ["client_id", "visitor_id", "visit_id"]

df_web_sorted["previous_step_num"] = df_web_sorted.groupby(group_cols)["step_num"].shift()
df_web_sorted["step_diff"] = df_web_sorted["step_num"] - df_web_sorted["previous_step_num"]

df_web_sorted["backtracked_event"] = df_web_sorted["step_diff"] < 0
df_web_sorted["jumped_event"] = df_web_sorted["step_diff"] > 1
df_web_sorted["repeated_step_event"] = df_web_sorted["step_diff"] == 0

In [17]:
# =========================
# Build session-level summary
# =========================

expected_sequence = "start > step_1 > step_2 > step_3 > confirm"

session_summary = (
    df_web_sorted
    .groupby(group_cols, as_index=False)
    .agg(
        session_start=("date_time", "min"),
        session_end=("date_time", "max"),
        n_events=("process_step", "size"),
        n_unique_steps=("process_step", "nunique"),
        step_sequence=("process_step", lambda x: " > ".join(x.astype(str))),
        started=("process_step", lambda x: (x == "start").any()),
        completed=("process_step", lambda x: (x == "confirm").any()),
        first_step=("process_step", "first"),
        last_step=("process_step", "last"),
        has_backtracking=("backtracked_event", "any"),
        has_step_jump=("jumped_event", "any"),
        has_repeated_step_event=("repeated_step_event", "any")
    )
)

session_summary["session_duration_min"] = (session_summary["session_end"] - session_summary["session_start"]).dt.total_seconds() / 60

session_summary["has_repeated_steps"] = session_summary["n_events"] > session_summary["n_unique_steps"]

session_summary["straight_through"] = session_summary["step_sequence"] == expected_sequence

session_summary["friction_flag"] = ~session_summary["straight_through"]

session_summary["incomplete_flag"] = ~session_summary["completed"]

session_summary["did_not_start_with_start"] = session_summary["first_step"] != "start"

session_summary["did_not_end_with_confirm"] = session_summary["last_step"] != "confirm"

session_summary.shape

(159112, 22)

In [18]:
# =========================
# Calculate completion time
# =========================

first_start = (
    df_web_sorted[df_web_sorted["process_step"] == "start"]
    .groupby(group_cols)["date_time"]
    .min()
    .reset_index(name="first_start_time")
)

first_confirm = (
    df_web_sorted[df_web_sorted["process_step"] == "confirm"]
    .groupby(group_cols)["date_time"]
    .min()
    .reset_index(name="first_confirm_time")
)

session_summary = (
    session_summary
    .merge(first_start, on=group_cols, how="left")
    .merge(first_confirm, on=group_cols, how="left")
)

session_summary["completion_time_min"] = (session_summary["first_confirm_time"] - session_summary["first_start_time"]).dt.total_seconds() / 60

session_summary.loc[~session_summary["completed"], "completion_time_min"] = np.nan

In [19]:
print("session_summary shape:", session_summary.shape)
print("Unique clients:", session_summary["client_id"].nunique())
print("Unique visits:", session_summary["visit_id"].nunique())

print("\nCompletion value counts:")
print(session_summary["completed"].value_counts(normalize=True).round(4))

print("\nStraight-through value counts:")
print(session_summary["straight_through"].value_counts(normalize=True).round(4))

print("\nFriction value counts:")
print(session_summary["friction_flag"].value_counts(normalize=True).round(4))

print("\nBacktracking value counts:")
print(session_summary["has_backtracking"].value_counts(normalize=True).round(4))

print("\nRepeated steps value counts:")
print(session_summary["has_repeated_steps"].value_counts(normalize=True).round(4))

print("\nStep jump value counts:")
print(session_summary["has_step_jump"].value_counts(normalize=True).round(4))

session_summary shape: (159112, 25)
Unique clients: 120157
Unique visits: 158095

Completion value counts:
completed
True     0.5696
False    0.4304
Name: proportion, dtype: float64

Straight-through value counts:
straight_through
False    0.7034
True     0.2966
Name: proportion, dtype: float64

Friction value counts:
friction_flag
True     0.7034
False    0.2966
Name: proportion, dtype: float64

Backtracking value counts:
has_backtracking
False    0.7497
True     0.2503
Name: proportion, dtype: float64

Repeated steps value counts:
has_repeated_steps
False    0.573
True     0.427
Name: proportion, dtype: float64

Step jump value counts:
has_step_jump
False    0.984
True     0.016
Name: proportion, dtype: float64


In [20]:
# =========================
# Create session-level big dataframe
# =========================

session_big_df = (
    session_summary
    .merge(df_exp, on="client_id", how="left", validate="many_to_one")
    .merge(df_demo, on="client_id", how="left", validate="many_to_one")
)

session_big_df.shape

(159112, 34)

In [21]:
# =========================
# Create session-level A/B dataframe
# =========================

session_ab_df = session_big_df[session_big_df["variation"].isin(["Test", "Control"])].copy()

print("session_big_df shape:", session_big_df.shape)
print("session_ab_df shape:", session_ab_df.shape)

print("\nSession counts by variation:")
print(session_ab_df["variation"].value_counts())

print("\nUnique clients by variation:")
print(session_ab_df.groupby("variation")["client_id"].nunique())

session_big_df shape: (159112, 34)
session_ab_df shape: (69447, 34)

Session counts by variation:
variation
Test       37204
Control    32243
Name: count, dtype: int64

Unique clients by variation:
variation
Control    23532
Test       26968
Name: client_id, dtype: int64


In [22]:
# =========================
# Session-level KPI summary
# =========================

session_ab_summary = (
    session_ab_df
    .groupby("variation")
    .agg(
        sessions=("visit_id", "count"),
        clients=("client_id", "nunique"),
        completion_rate=("completed", "mean"),
        straight_through_rate=("straight_through", "mean"),
        friction_rate=("friction_flag", "mean"),
        backtracking_rate=("has_backtracking", "mean"),
        repeated_steps_rate=("has_repeated_steps", "mean"),
        step_jump_rate=("has_step_jump", "mean"),
        wrong_start_rate=("did_not_start_with_start", "mean"),
        wrong_end_rate=("did_not_end_with_confirm", "mean"),
        median_session_duration_min=("session_duration_min", "median"),
        median_completion_time_min=("completion_time_min", "median")
    )
    .round(4)
)

session_ab_summary

,sessions,clients,completion_rate,straight_through_rate,friction_rate,backtracking_rate,repeated_steps_rate,step_jump_rate,wrong_start_rate,wrong_end_rate,median_session_duration_min,median_completion_time_min
variation,,,,,,,,,,,,
Control,32243,23532,0.4990,0.2941,0.7059,0.2002,0.3723,0.0110,0.0414,0.5190,2.6667,4.5167
Test,37204,26968,0.5857,0.2853,0.7147,0.2678,0.4479,0.0202,0.1115,0.4215,2.8000,3.9500


In [23]:
# =========================
# Save session-level dataframes
# =========================

session_big_df.to_csv(output_path / "session_big_df.csv", index=False)
session_ab_df.to_csv(output_path / "session_ab_df.csv", index=False)

## Session-Level DataFrames

The event-level web activity data was aggregated into a session-level dataset using `client_id`, `visitor_id`, and `visit_id`.

This allows us to evaluate the digital process at visit level instead of event level. The resulting DataFrame includes key behavioral indicators such as completion, backtracking, repeated steps, skipped steps, session duration, completion time, and friction.

The `session_ab_df` dataset keeps only Test and Control users, excluding the `empty` group. This DataFrame will be used as the main base for session-level A/B performance metrics.

In [24]:
# =========================
# Create clean vs friction session dataframes
# =========================

clean_sessions_df = session_ab_df[session_ab_df["straight_through"]].copy()
friction_sessions_df = session_ab_df[session_ab_df["friction_flag"]].copy()

print("Clean sessions shape:", clean_sessions_df.shape)
print("Friction sessions shape:", friction_sessions_df.shape)

print("\nClean sessions by variation:")
print(clean_sessions_df["variation"].value_counts())

print("\nFriction sessions by variation:")
print(friction_sessions_df["variation"].value_counts())

Clean sessions shape: (20095, 34)
Friction sessions shape: (49352, 34)

Clean sessions by variation:
variation
Test       10613
Control     9482
Name: count, dtype: int64

Friction sessions by variation:
variation
Test       26591
Control    22761
Name: count, dtype: int64


In [25]:
# =========================
# Validate clean + friction split
# =========================

total_split_sessions = len(clean_sessions_df) + len(friction_sessions_df)

print("Total A/B sessions:", len(session_ab_df))
print("Clean + friction sessions:", total_split_sessions)
print("Difference:", len(session_ab_df) - total_split_sessions)

Total A/B sessions: 69447
Clean + friction sessions: 69447
Difference: 0


In [26]:
# =========================
# Clean vs friction comparison by variation
# =========================

clean_friction_summary = (
    session_ab_df
    .groupby("variation")
    .agg(
        total_sessions=("visit_id", "count"),
        clean_sessions=("straight_through", "sum"),
        friction_sessions=("friction_flag", "sum"),
        clean_session_rate=("straight_through", "mean"),
        friction_session_rate=("friction_flag", "mean"),
        completed_sessions=("completed", "sum"),
        completion_rate=("completed", "mean"),
        backtracking_rate=("has_backtracking", "mean"),
        repeated_steps_rate=("has_repeated_steps", "mean"),
        step_jump_rate=("has_step_jump", "mean")
    )
    .round(4)
)

clean_friction_summary

,total_sessions,clean_sessions,friction_sessions,clean_session_rate,friction_session_rate,completed_sessions,completion_rate,backtracking_rate,repeated_steps_rate,step_jump_rate
variation,,,,,,,,,,
Control,32243,9482,22761,0.2941,0.7059,16088,0.4990,0.2002,0.3723,0.0110
Test,37204,10613,26591,0.2853,0.7147,21791,0.5857,0.2678,0.4479,0.0202


In [27]:
# =========================
# Friction reason breakdown
# =========================

friction_reason_summary = (
    session_ab_df
    .groupby("variation")
    .agg(
        incomplete_sessions=("incomplete_flag", "sum"),
        backtracking_sessions=("has_backtracking", "sum"),
        repeated_step_sessions=("has_repeated_steps", "sum"),
        step_jump_sessions=("has_step_jump", "sum"),
        wrong_start_sessions=("did_not_start_with_start", "sum"),
        wrong_end_sessions=("did_not_end_with_confirm", "sum")
    )
)

friction_reason_rates = (
    session_ab_df
    .groupby("variation")
    .agg(
        incomplete_rate=("incomplete_flag", "mean"),
        backtracking_rate=("has_backtracking", "mean"),
        repeated_step_rate=("has_repeated_steps", "mean"),
        step_jump_rate=("has_step_jump", "mean"),
        wrong_start_rate=("did_not_start_with_start", "mean"),
        wrong_end_rate=("did_not_end_with_confirm", "mean")
    )
    .round(4)
)

display(friction_reason_summary)
display(friction_reason_rates)

,incomplete_sessions,backtracking_sessions,repeated_step_sessions,step_jump_sessions,wrong_start_sessions,wrong_end_sessions
variation,,,,,,
Control,16155,6455,12003,355,1336,16735
Test,15413,9962,16664,753,4147,15681


,incomplete_rate,backtracking_rate,repeated_step_rate,step_jump_rate,wrong_start_rate,wrong_end_rate
variation,,,,,,
Control,0.5010,0.2002,0.3723,0.0110,0.0414,0.5190
Test,0.4143,0.2678,0.4479,0.0202,0.1115,0.4215


In [28]:
# =========================
# Save clean and friction session dataframes
# =========================

clean_sessions_df.to_csv(output_path / "clean_sessions_df.csv", index=False)
friction_sessions_df.to_csv(output_path / "friction_sessions_df.csv", index=False)

## Clean vs Friction Sessions

The session-level A/B dataset was split into two groups.

`clean_sessions_df` contains sessions that followed the ideal process sequence:

`start → step_1 → step_2 → step_3 → confirm`

These sessions represent users who completed the process smoothly without repeated steps, backtracking, skipped steps, or abandonment.

`friction_sessions_df` contains sessions that did not follow the ideal sequence. These sessions may include abandonment, repeated steps, backtracking, skipped steps, or other non-linear behavior.

The term friction is used instead of error because not every irregular journey is necessarily a technical problem. Some users may simply hesitate, review previous information, or leave the process voluntarily.

In [29]:
# =========================
# Build client-level behavior table
# =========================

client_behavior = (
    session_summary
    .groupby("client_id", as_index=False)
    .agg(
        total_sessions=("visit_id", "count"),
        total_events=("n_events", "sum"),
        completed_sessions=("completed", "sum"),
        started_sessions=("started", "sum"),
        straight_through_sessions=("straight_through", "sum"),
        friction_sessions=("friction_flag", "sum"),
        incomplete_sessions=("incomplete_flag", "sum"),
        backtracking_sessions=("has_backtracking", "sum"),
        repeated_step_sessions=("has_repeated_steps", "sum"),
        step_jump_sessions=("has_step_jump", "sum"),
        avg_session_duration_min=("session_duration_min", "mean"),
        median_session_duration_min=("session_duration_min", "median"),
        avg_completion_time_min=("completion_time_min", "mean"),
        median_completion_time_min=("completion_time_min", "median"),
        avg_events_per_session=("n_events", "mean")
    )
)

client_behavior.head()

,client_id,total_sessions,total_events,completed_sessions,started_sessions,straight_through_sessions,friction_sessions,incomplete_sessions,backtracking_sessions,repeated_step_sessions,step_jump_sessions,avg_session_duration_min,median_session_duration_min,avg_completion_time_min,median_completion_time_min,avg_events_per_session
0,169,1,5,1,1,1,0,0,0,0,0,3.550000,3.550000,3.550000,3.550000,5.0
1,336,1,2,0,1,0,1,1,0,1,0,15.800000,15.800000,NaN,NaN,2.0
2,546,1,5,1,1,1,0,0,0,0,0,2.216667,2.216667,2.216667,2.216667,5.0
3,555,1,5,1,1,1,0,0,0,0,0,2.633333,2.633333,2.633333,2.633333,5.0
4,647,1,5,1,1,1,0,0,0,0,0,6.283333,6.283333,6.283333,6.283333,5.0


In [30]:
# =========================
# Create client-level behavioral rates
# =========================

client_behavior["completion_rate"] = client_behavior["completed_sessions"] / client_behavior["total_sessions"]
client_behavior["started_rate"] = client_behavior["started_sessions"] / client_behavior["total_sessions"]
client_behavior["straight_through_rate"] = client_behavior["straight_through_sessions"] / client_behavior["total_sessions"]
client_behavior["friction_rate"] = client_behavior["friction_sessions"] / client_behavior["total_sessions"]
client_behavior["incomplete_rate"] = client_behavior["incomplete_sessions"] / client_behavior["total_sessions"]
client_behavior["backtracking_rate"] = client_behavior["backtracking_sessions"] / client_behavior["total_sessions"]
client_behavior["repeated_step_rate"] = client_behavior["repeated_step_sessions"] / client_behavior["total_sessions"]
client_behavior["step_jump_rate"] = client_behavior["step_jump_sessions"] / client_behavior["total_sessions"]

client_behavior["completed_any"] = client_behavior["completed_sessions"] > 0
client_behavior["straight_through_any"] = client_behavior["straight_through_sessions"] > 0
client_behavior["friction_any"] = client_behavior["friction_sessions"] > 0
client_behavior["backtracking_any"] = client_behavior["backtracking_sessions"] > 0
client_behavior["repeated_step_any"] = client_behavior["repeated_step_sessions"] > 0

client_behavior.head()

,client_id,total_sessions,total_events,completed_sessions,started_sessions,straight_through_sessions,friction_sessions,incomplete_sessions,backtracking_sessions,repeated_step_sessions,...,friction_rate,incomplete_rate,backtracking_rate,repeated_step_rate,step_jump_rate,completed_any,straight_through_any,friction_any,backtracking_any,repeated_step_any
0,169,1,5,1,1,1,0,0,0,0,...,0.0,0.0,0.0,0.0,0.0,True,True,False,False,False
1,336,1,2,0,1,0,1,1,0,1,...,1.0,1.0,0.0,1.0,0.0,False,False,True,False,True
2,546,1,5,1,1,1,0,0,0,0,...,0.0,0.0,0.0,0.0,0.0,True,True,False,False,False
3,555,1,5,1,1,1,0,0,0,0,...,0.0,0.0,0.0,0.0,0.0,True,True,False,False,False
4,647,1,5,1,1,1,0,0,0,0,...,0.0,0.0,0.0,0.0,0.0,True,True,False,False,False


In [31]:
# =========================
# Create client-level big dataframe
# =========================

client_big_df = (
    df_exp
    .merge(client_behavior, on="client_id", how="left", validate="one_to_one")
    .merge(df_demo, on="client_id", how="left", validate="one_to_one")
)

client_big_df.shape

(70609, 38)

In [32]:
# =========================
# Handle clients without web activity
# =========================

count_cols = [
    "total_sessions", "total_events", "completed_sessions", "started_sessions",
    "straight_through_sessions", "friction_sessions", "incomplete_sessions",
    "backtracking_sessions", "repeated_step_sessions", "step_jump_sessions"
]

rate_cols = [
    "completion_rate", "started_rate", "straight_through_rate", "friction_rate",
    "incomplete_rate", "backtracking_rate", "repeated_step_rate", "step_jump_rate"
]

bool_cols = [
    "completed_any", "straight_through_any", "friction_any",
    "backtracking_any", "repeated_step_any"
]

client_big_df[count_cols] = client_big_df[count_cols].fillna(0)
client_big_df[rate_cols] = client_big_df[rate_cols].fillna(0)
client_big_df[bool_cols] = client_big_df[bool_cols].fillna(False)

client_big_df["has_web_activity"] = client_big_df["total_sessions"] > 0

In [33]:
# =========================
# Create client-level A/B dataframe
# =========================

client_ab_df = client_big_df[client_big_df["variation"].isin(["Test", "Control"])].copy()

print("client_big_df shape:", client_big_df.shape)
print("client_ab_df shape:", client_ab_df.shape)

print("\nClients by variation:")
print(client_ab_df["variation"].value_counts())

print("\nClients with web activity by variation:")
print(client_ab_df.groupby("variation")["has_web_activity"].sum())

client_big_df shape: (70609, 39)
client_ab_df shape: (50500, 39)

Clients by variation:
variation
Test       26968
Control    23532
Name: count, dtype: int64

Clients with web activity by variation:
variation
Control    23532
Test       26968
Name: has_web_activity, dtype: int64


In [34]:
# =========================
# Client-level A/B summary
# =========================

client_ab_summary = (
    client_ab_df
    .groupby("variation")
    .agg(
        clients=("client_id", "nunique"),
        clients_with_web_activity=("has_web_activity", "sum"),
        avg_sessions_per_client=("total_sessions", "mean"),
        avg_events_per_client=("total_events", "mean"),
        completed_any_rate=("completed_any", "mean"),
        avg_completion_rate=("completion_rate", "mean"),
        straight_through_any_rate=("straight_through_any", "mean"),
        avg_straight_through_rate=("straight_through_rate", "mean"),
        friction_any_rate=("friction_any", "mean"),
        avg_friction_rate=("friction_rate", "mean"),
        backtracking_any_rate=("backtracking_any", "mean"),
        avg_backtracking_rate=("backtracking_rate", "mean"),
        repeated_step_any_rate=("repeated_step_any", "mean"),
        avg_repeated_step_rate=("repeated_step_rate", "mean"),
        median_session_duration_min=("median_session_duration_min", "median"),
        median_completion_time_min=("median_completion_time_min", "median")
    )
    .round(4)
)

client_ab_summary

,clients,clients_with_web_activity,avg_sessions_per_client,avg_events_per_client,completed_any_rate,avg_completion_rate,straight_through_any_rate,avg_straight_through_rate,friction_any_rate,avg_friction_rate,backtracking_any_rate,avg_backtracking_rate,repeated_step_any_rate,avg_repeated_step_rate,median_session_duration_min,median_completion_time_min
variation,,,,,,,,,,,,,,,,
Control,23532,23532,1.3702,5.9721,0.6559,0.5650,0.4026,0.3498,0.6938,0.6502,0.2582,0.2082,0.4225,0.3600,3.0333,4.5167
Test,26968,26968,1.3796,6.5522,0.6929,0.6245,0.3929,0.3407,0.7017,0.6593,0.3339,0.2744,0.5210,0.4492,3.1000,3.9500


In [35]:
# =========================
# Demographic balance check
# =========================

demographic_balance_summary = (
    client_ab_df
    .groupby("variation")
    .agg(
        clients=("client_id", "nunique"),
        avg_age=("client_age", "mean"),
        median_age=("client_age", "median"),
        avg_tenure_years=("client_tenure_years", "mean"),
        median_tenure_years=("client_tenure_years", "median"),
        avg_balance=("balance", "mean"),
        median_balance=("balance", "median"),
        avg_number_accounts=("number_accounts", "mean"),
        avg_calls_6_months=("calls_6_months", "mean"),
        avg_logons_6_months=("logons_6_months", "mean")
    )
    .round(2)
)

demographic_balance_summary

,clients,avg_age,median_age,avg_tenure_years,median_tenure_years,avg_balance,median_balance,avg_number_accounts,avg_calls_6_months,avg_logons_6_months
variation,,,,,,,,,,
Control,23532,47.50,48.5,12.09,11.0,150147.33,66024.18,2.26,3.13,6.17
Test,26968,47.16,47.5,11.98,11.0,148962.61,65468.36,2.25,3.06,6.10


In [36]:
# =========================
# Gender distribution by variation
# =========================

gender_balance = (
    pd.crosstab(
        client_ab_df["variation"],
        client_ab_df["gender"],
        normalize="index"
    )
    .round(4)
)

gender_balance

gender,Female,Male,Unknown
variation,,,
Control,0.3206,0.3388,0.3406
Test,0.3233,0.3330,0.3438


In [37]:
# =========================
# Save client-level dataframes
# =========================

client_big_df.to_csv(output_path / "client_big_df.csv", index=False)
client_ab_df.to_csv(output_path / "client_ab_df.csv", index=False)

## Client-Level DataFrames

The session-level behavior was aggregated into a client-level dataset.

`client_big_df` contains one row per client and combines experiment assignment, client profile information, and aggregated behavioral metrics across sessions.

`client_ab_df` keeps only Test and Control clients. This dataset is especially useful for client-level A/B testing because each client appears only once, avoiding the overrepresentation of highly active users.

These client-level datasets will be used for demographic analysis, experiment balance checks, client segmentation, and statistical comparisons between Test and Control.

In [39]:
# =========================
# Final dataframe shape summary
# =========================

final_shapes = pd.DataFrame({
    "dataframe": [
        "event_big_df",
        "event_experiment_df",
        "event_ab_df",
        "session_big_df",
        "session_ab_df",
        "clean_sessions_df",
        "friction_sessions_df",
        "client_big_df",
        "client_ab_df"
    ],
    "rows": [
        event_big_df.shape[0],
        event_experiment_df.shape[0],
        event_ab_df.shape[0],
        session_big_df.shape[0],
        session_ab_df.shape[0],
        clean_sessions_df.shape[0],
        friction_sessions_df.shape[0],
        client_big_df.shape[0],
        client_ab_df.shape[0]
    ],
    "columns": [
        event_big_df.shape[1],
        event_experiment_df.shape[1],
        event_ab_df.shape[1],
        session_big_df.shape[1],
        session_ab_df.shape[1],
        clean_sessions_df.shape[1],
        friction_sessions_df.shape[1],
        client_big_df.shape[1],
        client_ab_df.shape[1]
    ]
})

final_shapes

,dataframe,rows,columns
0,event_big_df,744641,14
1,event_experiment_df,443897,14
2,event_ab_df,317235,14
3,session_big_df,159112,34
4,session_ab_df,69447,34
5,clean_sessions_df,20095,34
6,friction_sessions_df,49352,34
7,client_big_df,70609,39
8,client_ab_df,50500,39


In [40]:
# =========================
# Final A/B sample checks
# =========================

print("Session-level A/B clients:")
print(session_ab_df.groupby("variation")["client_id"].nunique())

print("\nSession-level A/B sessions:")
print(session_ab_df["variation"].value_counts())

print("\nClient-level A/B clients:")
print(client_ab_df["variation"].value_counts())

print("\nClient-level clients with web activity:")
print(client_ab_df.groupby("variation")["has_web_activity"].sum())

Session-level A/B clients:
variation
Control    23532
Test       26968
Name: client_id, dtype: int64

Session-level A/B sessions:
variation
Test       37204
Control    32243
Name: count, dtype: int64

Client-level A/B clients:
variation
Test       26968
Control    23532
Name: count, dtype: int64

Client-level clients with web activity:
variation
Control    23532
Test       26968
Name: has_web_activity, dtype: int64


In [41]:
# =========================
# Check that A/B dataframes only include Test and Control
# =========================

print("session_ab_df variations:")
print(session_ab_df["variation"].value_counts(dropna=False))

print("\nclient_ab_df variations:")
print(client_ab_df["variation"].value_counts(dropna=False))

assert set(session_ab_df["variation"].dropna().unique()).issubset({"Test", "Control"})
assert set(client_ab_df["variation"].dropna().unique()).issubset({"Test", "Control"})

print("\nA/B files only contain Test and Control.")

session_ab_df variations:
variation
Test       37204
Control    32243
Name: count, dtype: int64

client_ab_df variations:
variation
Test       26968
Control    23532
Name: count, dtype: int64

A/B files only contain Test and Control.


In [42]:
# =========================
# Validate clean and friction split
# =========================

clean_sessions = len(clean_sessions_df)
friction_sessions = len(friction_sessions_df)
ab_sessions = len(session_ab_df)

print("Clean sessions:", clean_sessions)
print("Friction sessions:", friction_sessions)
print("Total clean + friction:", clean_sessions + friction_sessions)
print("Total A/B sessions:", ab_sessions)
print("Difference:", ab_sessions - (clean_sessions + friction_sessions))

assert clean_sessions + friction_sessions == ab_sessions

print("\nClean and friction sessions correctly cover all A/B sessions.")

Clean sessions: 20095
Friction sessions: 49352
Total clean + friction: 69447
Total A/B sessions: 69447
Difference: 0

Clean and friction sessions correctly cover all A/B sessions.


In [43]:
# =========================
# Final KPI comparison tables
# =========================

session_kpi_final = (
    session_ab_df
    .groupby("variation")
    .agg(
        sessions=("visit_id", "count"),
        clients=("client_id", "nunique"),
        completion_rate=("completed", "mean"),
        straight_through_rate=("straight_through", "mean"),
        friction_rate=("friction_flag", "mean"),
        backtracking_rate=("has_backtracking", "mean"),
        repeated_step_rate=("has_repeated_steps", "mean"),
        step_jump_rate=("has_step_jump", "mean"),
        median_session_duration_min=("session_duration_min", "median"),
        median_completion_time_min=("completion_time_min", "median")
    )
    .round(4)
)

client_kpi_final = (
    client_ab_df
    .groupby("variation")
    .agg(
        clients=("client_id", "nunique"),
        clients_with_web_activity=("has_web_activity", "sum"),
        completed_any_rate=("completed_any", "mean"),
        avg_completion_rate=("completion_rate", "mean"),
        straight_through_any_rate=("straight_through_any", "mean"),
        avg_straight_through_rate=("straight_through_rate", "mean"),
        friction_any_rate=("friction_any", "mean"),
        avg_friction_rate=("friction_rate", "mean"),
        backtracking_any_rate=("backtracking_any", "mean"),
        avg_backtracking_rate=("backtracking_rate", "mean"),
        repeated_step_any_rate=("repeated_step_any", "mean"),
        avg_repeated_step_rate=("repeated_step_rate", "mean")
    )
    .round(4)
)

display(session_kpi_final)
display(client_kpi_final)

,sessions,clients,completion_rate,straight_through_rate,friction_rate,backtracking_rate,repeated_step_rate,step_jump_rate,median_session_duration_min,median_completion_time_min
variation,,,,,,,,,,
Control,32243,23532,0.4990,0.2941,0.7059,0.2002,0.3723,0.0110,2.6667,4.5167
Test,37204,26968,0.5857,0.2853,0.7147,0.2678,0.4479,0.0202,2.8000,3.9500


,clients,clients_with_web_activity,completed_any_rate,avg_completion_rate,straight_through_any_rate,avg_straight_through_rate,friction_any_rate,avg_friction_rate,backtracking_any_rate,avg_backtracking_rate,repeated_step_any_rate,avg_repeated_step_rate
variation,,,,,,,,,,,,
Control,23532,23532,0.6559,0.5650,0.4026,0.3498,0.6938,0.6502,0.2582,0.2082,0.4225,0.3600
Test,26968,26968,0.6929,0.6245,0.3929,0.3407,0.7017,0.6593,0.3339,0.2744,0.5210,0.4492


In [53]:
# =========================
# Save final summary tables
# =========================

session_kpi_final.to_csv(output_path / "session_kpi_final.csv")
client_kpi_final.to_csv(output_path / "client_kpi_final.csv")
display(final_shapes)

,dataframe,rows,columns
0,event_big_df,744641,14
1,event_experiment_df,443897,14
2,event_ab_df,317235,14
3,session_big_df,159112,34
4,session_ab_df,69447,34
5,clean_sessions_df,20095,34
6,friction_sessions_df,49352,34
7,client_big_df,70609,39
8,client_ab_df,50500,39


In [45]:
# =========================
# Confirm saved files
# =========================

saved_files = sorted([file.name for file in output_path.glob("*.csv")])

print("Files saved in:", output_path)
print("\nSaved CSV files:")

for file in saved_files:
    print("-", file)

Files saved in: ..\1.2_Merged_Files_For_EDA

Saved CSV files:
- clean_sessions_df.csv
- client_ab_df.csv
- client_big_df.csv
- client_kpi_final.csv
- event_ab_df.csv
- event_big_df.csv
- event_experiment_df.csv
- final_dataframe_shapes.csv
- friction_sessions_df.csv
- session_ab_df.csv
- session_big_df.csv
- session_kpi_final.csv


## Final Output

This notebook created the final merged DataFrames required for the Vanguard A/B testing project.

The data was prepared at three main levels:

- Event level
- Session level
- Client level

The event-level data preserves the full digital journey and is useful for process-flow analysis and Tableau visualizations.

The session-level data is used to evaluate process performance, including completion, friction, repeated steps, backtracking, and session duration.

The client-level data aggregates behavior per client and is used for demographic analysis, experiment balance checks, and client-level A/B testing.

The `empty` group was excluded from the A/B datasets because it does not represent either the Test or Control group.

The final output files were saved in:

`1.2_Merged_Files_For_EDA/`

These files are now ready to be used for EDA, KPI analysis, hypothesis testing, Tableau dashboards, and the final project presentation.

In [46]:
session_ab_df = pd.read_csv("../1.2_Merged_Files_For_EDA/session_ab_df.csv")
client_ab_df = pd.read_csv("../1.2_Merged_Files_For_EDA/client_ab_df.csv")

In [52]:
display(clean_sessions_df.head())
display(client_ab_df.head())
display(client_big_df.head())
display(client_kpi_final.head())
display(event_ab_df.head())
display(event_big_df.head())
display(event_experiment_df.head())
display(friction_sessions_df.head())
display(session_ab_df.head())
display(session_big_df.head())
display(session_kpi_final.head())



,client_id,visitor_id,visit_id,session_start,session_end,n_events,n_unique_steps,step_sequence,started,completed,...,completion_time_min,variation,client_tenure_years,client_tenure_months,client_age,gender,number_accounts,balance,calls_6_months,logons_6_months
3,555,402506806_56087378777,637149525_38041617439_716659,2017-04-15 12:57:56,2017-04-15 13:00:34,5,5,start > step_1 > step_2 > step_3 > confirm,True,True,...,2.633333,Test,3.0,46.0,29.5,Unknown,2.0,25454.66,2.0,6.0
4,647,66758770_53988066587,40369564_40101682850_311847,2017-04-12 15:41:28,2017-04-12 15:47:45,5,5,start > step_1 > step_2 > step_3 > confirm,True,True,...,6.283333,Test,12.0,151.0,57.5,Male,2.0,30525.80,0.0,4.0
18,1195,766842522_69992551638,393817425_39015278493_996341,2017-04-05 20:15:26,2017-04-05 20:19:31,5,5,start > step_1 > step_2 > step_3 > confirm,True,True,...,4.083333,Control,21.0,262.0,54.5,Male,2.0,28457.96,2.0,5.0
21,1336,920624746_32603333901,583743392_96265099036_939815,2017-05-08 06:05:12,2017-05-08 06:08:43,5,5,start > step_1 > step_2 > step_3 > confirm,True,True,...,3.516667,Test,48.0,576.0,42.0,Male,4.0,130537.18,6.0,9.0
28,1516,182314299_63168583136,255400977_38039535960_779641,2017-04-06 22:14:24,2017-04-06 22:30:18,5,5,start > step_1 > step_2 > step_3 > confirm,True,True,...,15.900000,Test,12.0,150.0,58.5,Female,2.0,25408.39,5.0,8.0


,client_id,variation,total_sessions,total_events,completed_sessions,started_sessions,straight_through_sessions,friction_sessions,incomplete_sessions,backtracking_sessions,...,repeated_step_any,client_tenure_years,client_tenure_months,client_age,gender,number_accounts,balance,calls_6_months,logons_6_months,has_web_activity
0,9988021,Test,2,15,0,2,0,2,2,1,...,True,5.0,64.0,79.0,Unknown,2.0,189023.86,1.0,4.0,True
1,8320017,Test,1,5,1,1,1,0,0,0,...,False,22.0,274.0,34.5,Male,2.0,36001.90,5.0,8.0,True
2,4033851,Control,1,15,1,1,0,1,0,1,...,True,12.0,149.0,63.5,Male,2.0,142642.26,5.0,8.0,True
3,1982004,Test,1,5,1,1,1,0,0,0,...,False,6.0,80.0,44.5,Unknown,2.0,30231.76,1.0,4.0,True
4,9294070,Control,1,2,0,1,0,1,1,0,...,True,5.0,70.0,29.0,Unknown,2.0,34254.54,0.0,3.0,True


,client_id,variation,total_sessions,total_events,completed_sessions,started_sessions,straight_through_sessions,friction_sessions,incomplete_sessions,backtracking_sessions,...,repeated_step_any,client_tenure_years,client_tenure_months,client_age,gender,number_accounts,balance,calls_6_months,logons_6_months,has_web_activity
0,9988021,Test,2,15,0,2,0,2,2,1,...,True,5.0,64.0,79.0,Unknown,2.0,189023.86,1.0,4.0,True
1,8320017,Test,1,5,1,1,1,0,0,0,...,False,22.0,274.0,34.5,Male,2.0,36001.90,5.0,8.0,True
2,4033851,Control,1,15,1,1,0,1,0,1,...,True,12.0,149.0,63.5,Male,2.0,142642.26,5.0,8.0,True
3,1982004,Test,1,5,1,1,1,0,0,0,...,False,6.0,80.0,44.5,Unknown,2.0,30231.76,1.0,4.0,True
4,9294070,Control,1,2,0,1,0,1,1,0,...,True,5.0,70.0,29.0,Unknown,2.0,34254.54,0.0,3.0,True


,clients,clients_with_web_activity,completed_any_rate,avg_completion_rate,straight_through_any_rate,avg_straight_through_rate,friction_any_rate,avg_friction_rate,backtracking_any_rate,avg_backtracking_rate,repeated_step_any_rate,avg_repeated_step_rate
variation,,,,,,,,,,,,
Control,23532,23532,0.6559,0.5650,0.4026,0.3498,0.6938,0.6502,0.2582,0.2082,0.4225,0.3600
Test,26968,26968,0.6929,0.6245,0.3929,0.3407,0.7017,0.6593,0.3339,0.2744,0.5210,0.4492


,client_id,visitor_id,visit_id,process_step,date_time,variation,client_tenure_years,client_tenure_months,client_age,gender,number_accounts,balance,calls_6_months,logons_6_months
0,9988021,580560515_7732621733,781255054_21935453173_531117,step_3,2017-04-17 15:27:07,Test,5.0,64.0,79.0,Unknown,2.0,189023.86,1.0,4.0
1,9988021,580560515_7732621733,781255054_21935453173_531117,step_2,2017-04-17 15:26:51,Test,5.0,64.0,79.0,Unknown,2.0,189023.86,1.0,4.0
2,9988021,580560515_7732621733,781255054_21935453173_531117,step_3,2017-04-17 15:19:22,Test,5.0,64.0,79.0,Unknown,2.0,189023.86,1.0,4.0
3,9988021,580560515_7732621733,781255054_21935453173_531117,step_2,2017-04-17 15:19:13,Test,5.0,64.0,79.0,Unknown,2.0,189023.86,1.0,4.0
4,9988021,580560515_7732621733,781255054_21935453173_531117,step_3,2017-04-17 15:18:04,Test,5.0,64.0,79.0,Unknown,2.0,189023.86,1.0,4.0


...

,client_id,visitor_id,visit_id,process_step,date_time,variation,client_tenure_years,client_tenure_months,client_age,gender,number_accounts,balance,calls_6_months,logons_6_months
0,9988021,580560515_7732621733,781255054_21935453173_531117,step_3,2017-04-17 15:27:07,Test,5.0,64.0,79.0,Unknown,2.0,189023.86,1.0,4.0
1,9988021,580560515_7732621733,781255054_21935453173_531117,step_2,2017-04-17 15:26:51,Test,5.0,64.0,79.0,Unknown,2.0,189023.86,1.0,4.0
2,9988021,580560515_7732621733,781255054_21935453173_531117,step_3,2017-04-17 15:19:22,Test,5.0,64.0,79.0,Unknown,2.0,189023.86,1.0,4.0
3,9988021,580560515_7732621733,781255054_21935453173_531117,step_2,2017-04-17 15:19:13,Test,5.0,64.0,79.0,Unknown,2.0,189023.86,1.0,4.0
4,9988021,580560515_7732621733,781255054_21935453173_531117,step_3,2017-04-17 15:18:04,Test,5.0,64.0,79.0,Unknown,2.0,189023.86,1.0,4.0


,client_id,visitor_id,visit_id,session_start,session_end,n_events,n_unique_steps,step_sequence,started,completed,...,completion_time_min,variation,client_tenure_years,client_tenure_months,client_age,gender,number_accounts,balance,calls_6_months,logons_6_months
11,934,810392784_45004760546,7076463_57954418406_971348,2017-04-18 02:36:30,2017-04-18 02:38:52,4,1,start > start > start > start,True,False,...,NaN,Test,9.0,109.0,51.0,Female,2.0,32522.88,0.0,3.0
12,1028,42237450_62128060588,557292053_87239438319_391157,2017-04-08 18:51:28,2017-04-08 19:00:26,9,4,start > step_1 > step_1 > step_2 > step_3 > st...,True,False,...,NaN,Control,12.0,145.0,36.0,Male,3.0,103520.22,1.0,4.0
14,1104,194240915_18158000533,543158812_46395476577_767725,2017-06-12 07:49:18,2017-06-12 07:49:18,1,1,start,True,False,...,NaN,Control,5.0,66.0,48.0,Unknown,3.0,154643.94,6.0,9.0
15,1104,194240915_18158000533,643221571_99977972121_69283,2017-06-20 22:31:33,2017-06-20 22:31:33,1,1,start,True,False,...,NaN,Control,5.0,66.0,48.0,Unknown,3.0,154643.94,6.0,9.0
16,1186,446844663_31615102958,507052512_11309370126_442139,2017-04-08 15:59:16,2017-04-08 15:59:16,1,1,start,True,False,...,NaN,Control,8.0,99.0,22.0,Unknown,2.0,31662.52,0.0,3.0


,client_id,visitor_id,visit_id,session_start,session_end,n_events,n_unique_steps,step_sequence,started,completed,...,completion_time_min,variation,client_tenure_years,client_tenure_months,client_age,gender,number_accounts,balance,calls_6_months,logons_6_months
0,555,402506806_56087378777,637149525_38041617439_716659,2017-04-15 12:57:56,2017-04-15 13:00:34,5,5,start > step_1 > step_2 > step_3 > confirm,True,True,...,2.633333,Test,3.0,46.0,29.5,Unknown,2.0,25454.66,2.0,6.0
1,647,66758770_53988066587,40369564_40101682850_311847,2017-04-12 15:41:28,2017-04-12 15:47:45,5,5,start > step_1 > step_2 > step_3 > confirm,True,True,...,6.283333,Test,12.0,151.0,57.5,Male,2.0,30525.80,0.0,4.0
2,934,810392784_45004760546,7076463_57954418406_971348,2017-04-18 02:36:30,2017-04-18 02:38:52,4,1,start > start > start > start,True,False,...,NaN,Test,9.0,109.0,51.0,Female,2.0,32522.88,0.0,3.0
3,1028,42237450_62128060588,557292053_87239438319_391157,2017-04-08 18:51:28,2017-04-08 19:00:26,9,4,start > step_1 > step_1 > step_2 > step_3 > st...,True,False,...,NaN,Control,12.0,145.0,36.0,Male,3.0,103520.22,1.0,4.0
4,1104,194240915_18158000533,543158812_46395476577_767725,2017-06-12 07:49:18,2017-06-12 07:49:18,1,1,start,True,False,...,NaN,Control,5.0,66.0,48.0,Unknown,3.0,154643.94,6.0,9.0


,client_id,visitor_id,visit_id,session_start,session_end,n_events,n_unique_steps,step_sequence,started,completed,...,completion_time_min,variation,client_tenure_years,client_tenure_months,client_age,gender,number_accounts,balance,calls_6_months,logons_6_months
0,169,201385055_71273495308,749567106_99161211863_557568,2017-04-12 20:19:36,2017-04-12 20:23:09,5,5,start > step_1 > step_2 > step_3 > confirm,True,True,...,3.550000,empty,21.0,262.0,47.5,Male,2.0,501570.72,4.0,4.0
1,336,64757908_3400128256,649044751_80905125055_554468,2017-06-01 07:26:55,2017-06-01 07:42:43,2,1,start > start,True,False,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,546,475037402_89828530214,731811517_9330176838_94847,2017-06-17 10:03:29,2017-06-17 10:05:42,5,5,start > step_1 > step_2 > step_3 > confirm,True,True,...,2.216667,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,555,402506806_56087378777,637149525_38041617439_716659,2017-04-15 12:57:56,2017-04-15 13:00:34,5,5,start > step_1 > step_2 > step_3 > confirm,True,True,...,2.633333,Test,3.0,46.0,29.5,Unknown,2.0,25454.66,2.0,6.0
4,647,66758770_53988066587,40369564_40101682850_311847,2017-04-12 15:41:28,2017-04-12 15:47:45,5,5,start > step_1 > step_2 > step_3 > confirm,True,True,...,6.283333,Test,12.0,151.0,57.5,Male,2.0,30525.80,0.0,4.0


,sessions,clients,completion_rate,straight_through_rate,friction_rate,backtracking_rate,repeated_step_rate,step_jump_rate,median_session_duration_min,median_completion_time_min
variation,,,,,,,,,,
Control,32243,23532,0.4990,0.2941,0.7059,0.2002,0.3723,0.0110,2.6667,4.5167
Test,37204,26968,0.5857,0.2853,0.7147,0.2678,0.4479,0.0202,2.8000,3.9500
